In [ ]:
# 1. 安装 Kaggle CLI
!pip install kaggle -q

# 2. 上传你的 kaggle.json（从 Kaggle → Account → Create API Token 下载）
from google.colab import files
files.upload()  # 选择你的 kaggle.json

# 3. 配置权限
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. 下载数据集
!kaggle datasets download -d shubhamkarande13/d-fire

# 5. 解压到指定文件夹
!unzip d-fire.zip -d /content/D-Fire

In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/DiffPure_extracted/DiffPure-master')  # 换成你实际放purify_bld_fixed.py的目录
import purify_bld_fixed as bld

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
detector = YOLO('/content/drive/MyDrive/dfire_checkpoints/yolov8n_dfire_detector.pt')

In [ ]:
CKPT_AE   = "/content/drive/MyDrive/bld_purify_checkpoints/bld_autoencoder.pt"
CKPT_DIFF = "/content/drive/MyDrive/bld_purify_checkpoints/bld_diffusion.pt"
import torch
ae = bld.BinaryAutoEncoder().to(bld.DEVICE)
ae.load_state_dict(torch.load(CKPT_AE, map_location=bld.DEVICE))
ae.eval()

diff_model = bld.ReverseModel(D=ae.D).to(bld.DEVICE)
diff_model.load_state_dict(torch.load(CKPT_DIFF, map_location=bld.DEVICE))
diff_model.eval()

betas, flip_prob = bld.get_schedule()
print("AE + diffusion 权重加载完毕")

In [ ]:
import torch
from ultralytics.utils.loss import v8DetectionLoss
detector.model = detector.model.to(bld.DEVICE)
detector.model.train()   # 切train模式，loss才会被正确计算（但不会更新权重，因为我们不会调用optimizer.step）
attack_yolo = YOLO('/content/drive/MyDrive/dfire_checkpoints/yolov8n_dfire_detector.pt')
attack_yolo.model.to(bld.DEVICE)          # ← 补上这一行
attack_yolo.model.train()
for p in attack_yolo.model.parameters():
    p.requires_grad_(True)

criterion = v8DetectionLoss(attack_yolo.model)


def pgd_attack(model, criterion, x, targets, eps=8/255, alpha=2/255, steps=10):
    """
    model   : detector.model (DetectionModel)
    x       : (B,3,H,W) float [0,1]，注意要 resize 到训练时的 imgsz=384
    targets : dict，需包含 'cls', 'bboxes', 'batch_idx' 三个key（YOLO格式）
    """
    x_adv = (x + torch.empty_like(x).uniform_(-eps, eps)).clamp(0, 1).requires_grad_(True)
    for _ in range(steps):
        preds = model(x_adv)
        loss, _ = criterion(preds, targets)
        loss = loss.sum()
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = (x_adv.detach() + alpha * grad.sign()).clamp(x - eps, x + eps).clamp(0, 1)
        x_adv.requires_grad_(True)
    return x_adv.detach()

print(criterion)
print(type(criterion))

In [ ]:
import glob
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

class DFireDetDataset(Dataset):
    def __init__(self, root, split='test', size=384):
        self.img_paths = sorted(glob.glob(f'{root}/{split}/images/*.jpg'))
        self.label_dir = f'{root}/{split}/labels'
        self.tf = T.Compose([T.Resize((size, size)), T.ToTensor()])

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        p = self.img_paths[idx]
        x = self.tf(Image.open(p).convert('RGB'))
        lp = os.path.join(self.label_dir, os.path.basename(p).rsplit('.', 1)[0] + '.txt')
        boxes = []
        if os.path.exists(lp):
            with open(lp) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) == 5:
                        boxes.append([float(v) for v in parts])
        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 5))
        return x, boxes, p

def det_collate(batch):
    imgs, boxes_list, paths = zip(*batch)
    cls_l, bbox_l, bidx_l = [], [], []
    for i, b in enumerate(boxes_list):
        if b.shape[0] > 0:
            cls_l.append(b[:, 0:1]); bbox_l.append(b[:, 1:5])
            bidx_l.append(torch.full((b.shape[0],), i))
    targets = {
        'cls':       torch.cat(cls_l, 0)  if cls_l  else torch.zeros((0, 1)),
        'bboxes':    torch.cat(bbox_l, 0) if bbox_l else torch.zeros((0, 4)),
        'batch_idx': torch.cat(bidx_l, 0) if bidx_l else torch.zeros((0,)),
    }
    return torch.stack(imgs), targets, paths

det_test_ds = DFireDetDataset(bld.DFIRE_ROOT, split='test', size=384)
det_test_loader = DataLoader(det_test_ds, batch_size=8, shuffle=False, collate_fn=det_collate)

In [ ]:
import shutil

def save_as_dfire_split(x_batch, paths, save_dir):
    os.makedirs(f"{save_dir}/images", exist_ok=True)
    os.makedirs(f"{save_dir}/labels", exist_ok=True)
    for img, p in zip(x_batch, paths):
        name = os.path.basename(p).rsplit('.', 1)[0]
        T.ToPILImage()(img.cpu()).save(f"{save_dir}/images/{name}.jpg")
        lp = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        if os.path.exists(lp):
            shutil.copy(lp, f"{save_dir}/labels/{name}.txt")

def eval_map(save_dir, img_size=384):
    yaml_path = f"{save_dir}/tmp.yaml"
    with open(yaml_path, 'w') as f:
        yaml.dump({'path': save_dir, 'train': 'images', 'val': 'images',
                   'names': {0: 'fire', 1: 'smoke'}}, f)
    m = detector.val(data=yaml_path, imgsz=img_size, verbose=False)
    return m.box.map50, m.box.map

In [ ]:
"""
Cell 7 — 完整测试集评测（断点续跑版）
依赖：ae, diff_model, betas, flip_prob, attack_yolo, criterion, detector,
      pgd_attack, DFireDetDataset, det_collate, save_as_dfire_split, eval_map
      (这些应该都已经在你的Colab session里定义好了，来自 Cell1~6)
"""

import os
import yaml
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

# ---------------- 配置 ----------------
N_EVAL   = None          # None = 跑完整test set；先设成比如300做个中等规模验证也可以
T_STAR   = 40             # 先用单一t*跑全量，确认流程稳定后再考虑扫描多个t*
BATCH    = 8
OUT_ROOT = '/content/drive/MyDrive/dfire_adv_eval'   # 存Drive，断连不丢

for split in OUTPUT_SPLITS:
    os.makedirs(f'{OUT_ROOT}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUT_ROOT}/{split}/labels', exist_ok=True)

progress_path = f'{OUT_ROOT}/done.txt'
done = set()
if os.path.exists(progress_path):
    done = set(open(progress_path).read().split())
print(f"已完成 {len(done)} 张图（断点续跑会自动跳过它们）")

det_ds = DFireDetDataset(bld.DFIRE_ROOT, split='test', size=384)
if N_EVAL:
    det_ds.img_paths = det_ds.img_paths[:N_EVAL]
det_loader = DataLoader(det_ds, batch_size=BATCH, shuffle=False, collate_fn=det_collate)
print(f"本次评测图片总数: {len(det_ds)}")

# ---------------- 主循环：逐batch攻击 + 净化 + 存盘 ----------------
progress_f = open(progress_path, 'a')

for x, targets, paths in tqdm(det_loader, desc="attack+purify"):
    names = [os.path.basename(p) for p in paths]
    if all(n in done for n in names):
        continue   # 这个batch全部处理过，跳过

    x = x.to(bld.DEVICE)

    # 有些图是background（没有框），PGD无法针对空targets求loss，直接跳过攻击，
    # 但仍然要保留这些图参与最终mAP统计（它们贡献recall/precision里的负样本部分）
    if targets['bboxes'].shape[0] == 0:
        x_adv = x.clone()
    else:
        targets_gpu = {k: v.to(bld.DEVICE) for k, v in targets.items()}
        x_adv = pgd_attack(attack_yolo.model, criterion, x, targets_gpu)

    # resize到AE训练分辨率(128)做净化，再resize回384给检测器用
    x_adv_128 = F.interpolate(x_adv, size=128, mode='bilinear', align_corners=False)
    with torch.no_grad():
        z_adv = ae.encode_binary(x_adv_128)
        t_vec = torch.full((z_adv.shape[0],), T_STAR - 1,
                            device=bld.DEVICE, dtype=torch.long)
        z_noised = bld.q_sample(z_adv, t_vec, flip_prob)
        z_rec    = bld.reverse(diff_model, z_noised, T_STAR, betas, flip_prob)
        x_pur_128 = ae.decode(z_rec.float())
    x_pur = F.interpolate(x_pur_128, size=384, mode='bilinear', align_corners=False)

    save_as_dfire_split(x,       paths, f'{OUT_ROOT}/clean')
    save_as_dfire_split(x_adv,   paths, f'{OUT_ROOT}/adv')
    save_as_dfire_split(x_pur,   paths, f'{OUT_ROOT}/purified')

    for n in names:
        progress_f.write(n + '\n')
    progress_f.flush()
    done.update(names)

progress_f.close()
print(f"\n全部处理完成，共 {len(done)} 张图已存入 {OUT_ROOT}")


# ---------------- 全量mAP对比 ----------------
print("\n=== 完整测试集 mAP 对比 ===")
results = {}
for name in ['clean', 'adv', 'purified']:
    map50, map5095 = eval_map(f'{OUT_ROOT}/{name}', img_size=384)
    results[name] = (map50, map5095)
    print(f"{name:10s}  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")

recovery = (results['purified'][0] - results['adv'][0]) / max(results['clean'][0] - results['adv'][0], 1e-8)
print(f"\n净化恢复比例 (purified-adv)/(clean-adv) = {recovery:.1%}")

In [ ]:
"""
在AE的Encoder/Decoder里，32x32分辨率处各新增一层self-attention
（原模型只在16x16的bottleneck有attention，这里在更早的分辨率补一层）
从 ae_taskaware_finetuned.pt 手动搬运可复用的权重，只有新插入的attention层是随机初始化
依赖：bld模块 (BASE_CH=32, CH_MULT_ENC=(1,2,4), LATENT_CHANNELS=8, NUM_RES_BLOCKS=2)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

ResnetBlock = bld.ResnetBlock
AttnBlock   = bld.AttnBlock
Downsample  = bld.Downsample
Upsample    = bld.Upsample
Normalize   = bld.Normalize


# ---------------- 1. 新Encoder：在32x32(mul=4阶段的两个ResnetBlock之后、Downsample之前)插入attention ----------------
class EncoderAttn(nn.Module):
    def __init__(self, in_channels=3, base_ch=bld.BASE_CH,
                 num_res_blocks=bld.NUM_RES_BLOCKS,
                 latent_channels=bld.LATENT_CHANNELS,
                 ch_mult=bld.CH_MULT_ENC):
        super().__init__()
        layers = [nn.Conv2d(in_channels, base_ch, 3, padding=1)]      # idx0 stem
        in_ch = base_ch
        for stage_i, mul in enumerate(ch_mult):
            out_ch = base_ch * mul
            for _ in range(num_res_blocks):
                layers.append(ResnetBlock(in_ch, out_ch))
                in_ch = out_ch
            if stage_i == len(ch_mult) - 1:        # 最后一个stage(mul=4)，在Downsample前插入新attention
                layers.append(AttnBlock(in_ch))     # ← 新增：32x32分辨率
            layers.append(Downsample(in_ch))
        layers += [
            ResnetBlock(in_ch),
            AttnBlock(in_ch),                       # 原有的16x16 bottleneck attention
            ResnetBlock(in_ch),
            Normalize(in_ch),
            nn.Conv2d(in_ch, latent_channels, 3, padding=1),
        ]
        self.conv_layers = nn.Sequential(*layers)

    def forward(self, x, hard=True):
        logits = self.conv_layers(x)
        if not hard:
            return torch.sigmoid(logits)
        z_soft = torch.sigmoid(logits)
        z_hard = (z_soft > 0.5).float()
        return z_soft + (z_hard - z_soft).detach()


# ---------------- 2. 新Decoder：对称地在32x32处插入attention ----------------
class DecoderAttn(nn.Module):
    def __init__(self, out_channels=3, base_ch=bld.BASE_CH,
                 num_res_blocks=bld.NUM_RES_BLOCKS,
                 latent_channels=bld.LATENT_CHANNELS,
                 ch_mult=bld.CH_MULT_ENC):
        super().__init__()
        ch_mult_dec = tuple(reversed(ch_mult))
        in_ch = base_ch * ch_mult[-1]
        layers = [
            nn.Conv2d(latent_channels, in_ch, 3, padding=1),
            ResnetBlock(in_ch),
            AttnBlock(in_ch),                       # 原有的16x16 bottleneck attention
            ResnetBlock(in_ch),
        ]
        for stage_i, mul in enumerate(ch_mult_dec):
            out_ch = base_ch * mul
            for _ in range(num_res_blocks):
                layers.append(ResnetBlock(in_ch, out_ch))
                in_ch = out_ch
            layers.append(Upsample(in_ch))
            if stage_i == 0:                        # 对应encoder mul=4阶段，镜像插入
                layers.append(AttnBlock(in_ch))      # ← 新增：32x32分辨率
        layers += [
            Normalize(in_ch),
            nn.Conv2d(in_ch, out_channels, 3, padding=1),
            nn.Sigmoid(),
        ]
        self.conv_layers = nn.Sequential(*layers)

    def forward(self, z):
        return self.conv_layers(z)


class BinaryAutoEncoderAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderAttn()
        self.decoder = DecoderAttn()
        self.latent_h = bld.IMG_SIZE // (2 ** len(bld.CH_MULT_ENC))
        self.latent_w = self.latent_h
        self.D = bld.LATENT_CHANNELS * self.latent_h * self.latent_w

    def encode_binary(self, x):
        with torch.no_grad():
            z = self.encoder(x, hard=True)
        return z.long().view(z.shape[0], -1)

    def decode(self, z_flat):
        B = z_flat.shape[0]
        z = z_flat.view(B, bld.LATENT_CHANNELS, self.latent_h, self.latent_w)
        return self.decoder(z)

    def forward(self, x):
        z = self.encoder(x, hard=True)
        z_flat = z.view(z.shape[0], -1)
        x_rec = self.decode(z_flat)
        return x_rec, z


# ---------------- 3. 手动搬运权重：老模型的每一层 -> 新模型对应层，新插入的attention跳过（随机初始化） ----------------
def transplant_encoder(old_enc, new_enc):
    # 老encoder的conv_layers索引: 0=stem,1-2=stage1(mul1),3=down,4-5=stage2(mul2),6=down,
    #                            7-8=stage3(mul4),9=down,10=res,11=attn(bottleneck),12=res,13=norm,14=conv_out
    # 新encoder多插入了一层attn在旧index8和9之间，之后所有index整体+1
    old_map = list(range(9))      # 0..8 不变
    new_map = list(range(9))
    old_map += list(range(9, 15)) # 老的 9..14
    new_map += list(range(10, 16))# 对应新的 10..15 (因为9号位置插入了新attn)
    for oi, ni in zip(old_map, new_map):
        new_enc.conv_layers[ni].load_state_dict(old_enc.conv_layers[oi].state_dict())
    print(f"Encoder: 搬运了 {len(old_map)} 层，新增1层attention(32x32)为随机初始化")

def transplant_decoder(old_dec, new_dec):
    # 老decoder索引: 0=conv_in,1=res,2=attn(bottleneck),3=res,
    #               4-5=mul4 stage res,6=upsample,
    #               7-8=mul2 stage res,9=upsample,
    #               10-11=mul1 stage res,12=upsample,13=norm,14=conv_out,15=sigmoid
    # 新decoder在老index6(第一次upsample)之后插入新attn，之后所有index整体+1
    old_map = list(range(7))       # 0..6 不变
    new_map = list(range(7))
    old_map += list(range(7, 16))  # 老的 7..15
    new_map += list(range(8, 17))  # 对应新的 8..16
    for oi, ni in zip(old_map, new_map):
        new_dec.conv_layers[ni].load_state_dict(old_dec.conv_layers[oi].state_dict())
    print(f"Decoder: 搬运了 {len(old_map)} 层，新增1层attention(32x32)为随机初始化")


# ---------------- 4. 执行热启动 ----------------
old_ae = bld.BinaryAutoEncoder().to(bld.DEVICE)
old_ae.load_state_dict(
    torch.load('/content/drive/MyDrive/bld_purify_checkpoints/ae_taskaware_finetuned.pt',
               map_location=bld.DEVICE)
)

ae_attn = BinaryAutoEncoderAttn().to(bld.DEVICE)
transplant_encoder(old_ae.encoder, ae_attn.encoder)
transplant_decoder(old_ae.decoder, ae_attn.decoder)

# 关键修复：新插入的attention层(encoder idx9, decoder idx7)默认初始化不是恒等映射，
# 会在训练一开始就对热启动权重造成随机扰动。把proj层清零，让它们从"完全不改变输入"开始训练，
# 后续训练是纯增量式改进，不用先花epoch去抵消插入造成的干扰。
nn.init.zeros_(ae_attn.encoder.conv_layers[9].proj.weight)
nn.init.zeros_(ae_attn.encoder.conv_layers[9].proj.bias)
nn.init.zeros_(ae_attn.decoder.conv_layers[7].proj.weight)
nn.init.zeros_(ae_attn.decoder.conv_layers[7].proj.bias)
print("新增attention层已zero-init，训练起点等价于恒等映射")

print("热启动完成，ae_attn 已就绪，可以开始微调训练")

In [ ]:
# ==============================================================================
# 【旧版 GCN 定义｜手写固定层级｜仅供参考】
# 这是早期 EncoderGCN / DecoderGCN / BinaryAutoEncoderGCN 实现。
# 最终训练与 mAP 使用后面的“新版 GCN 主定义”单元；正常运行时请跳过本单元。
# ==============================================================================

"""
在 ae_attn 基础上，给 Decoder 的 64x64 / 32x32 两个中间层级各插入一个 GCN skip block：
  - 邻接矩阵 A：由 Encoder 对称层特征，经 self-attention 式 QK 变换后 softmax 得到
  - 特征矩阵 X：Decoder 当前层的特征（不是 encoder 的）
  - 图卷积核心运算：A @ X @ W，zero-init 残差接回 decoder 主干

刻意跳过最外层(128x128,几乎是原始像素)和bottleneck层(16x16,会超出2048bit预算)，
只在中间两个层级做skip，避免绕过净化框架 / 突破比特预算这两个问题。

范围限制：本次只验证clean图像下ae_only的mAP，不接PGD攻击/净化，先确认架构本身是否有效。
依赖：bld模块, ae_attn (来自 ae_add_attention_32x32.py, 已经完成 det_loss+attn 两阶段训练)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

ResnetBlock = bld.ResnetBlock
AttnBlock   = bld.AttnBlock
Downsample  = bld.Downsample
Upsample    = bld.Upsample
Normalize   = bld.Normalize


# ---------------- 1. GCN skip block ----------------
class GraphConvSkipBlock(nn.Module):
    """
    邻接矩阵完全由 encoder 特征算出（self-attention式 QK，无V —— 只要结构关系，不要encoder的内容值）
    特征矩阵是 decoder 自己当前层的特征
    H' = A @ X @ W， zero-init残差接回，起点=恒等映射
    """
    def __init__(self, channels):
        super().__init__()
        self.norm_enc = Normalize(channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.gcn_w = nn.Linear(channels, channels)
        self.proj = nn.Conv2d(channels, channels, 1)

        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x_dec, feat_enc):
        B, C, H, W = x_dec.shape
        h_enc = self.norm_enc(feat_enc)
        q = self.q(h_enc).reshape(B, C, -1).permute(0, 2, 1)     # (B,N,C)
        k = self.k(h_enc).reshape(B, C, -1)                       # (B,C,N)
        adj = torch.softmax(torch.bmm(q, k) * C**-0.5, dim=-1)    # (B,N,N) 邻接矩阵，softmax起到度归一化作用

        x_flat = x_dec.reshape(B, C, -1).permute(0, 2, 1)         # (B,N,C) 特征矩阵 X
        agg = torch.bmm(adj, x_flat)                               # A @ X
        agg = self.gcn_w(agg)                                      # A @ X @ W
        agg = agg.permute(0, 2, 1).reshape(B, C, H, W)

        return x_dec + self.proj(agg)


# ---------------- 2. Encoder：手动forward，暴露64x64/32x32中间特征 ----------------
import copy

class EncoderGCN(nn.Module):
    def __init__(self, base_encoder_attn):
        super().__init__()
        old = list(base_encoder_attn.conv_layers)
        self.stem   = copy.deepcopy(old[0])
        self.stage1 = copy.deepcopy(nn.Sequential(*old[1:3]))
        self.down1  = copy.deepcopy(old[3])
        self.stage2 = copy.deepcopy(nn.Sequential(*old[4:6]))
        self.down2  = copy.deepcopy(old[6])
        self.stage3 = copy.deepcopy(nn.Sequential(*old[7:9]))
        self.attn32 = copy.deepcopy(old[9])
        self.down3  = copy.deepcopy(old[10])
        self.bottleneck = copy.deepcopy(nn.Sequential(*old[11:14]))
        self.norm_out = copy.deepcopy(old[14])
        self.conv_out  = copy.deepcopy(old[15])
    # forward部分不变

    def forward(self, x, hard=True):
        h = self.stem(x)
        h = self.stage1(h)
        h = self.down1(h)
        h = self.stage2(h)
        feat_64 = h                                 # 暴露给decoder做邻接矩阵
        h = self.down2(h)
        h = self.stage3(h)
        feat_32 = h                                 # 暴露给decoder做邻接矩阵
        h = self.attn32(h)
        h = self.down3(h)
        h = self.bottleneck(h)
        h = self.norm_out(h)
        logits = self.conv_out(h)

        if not hard:
            z = torch.sigmoid(logits)
        else:
            z_soft = torch.sigmoid(logits)
            z_hard = (z_soft > 0.5).float()
            z = z_soft + (z_hard - z_soft).detach()
        return z, feat_64, feat_32


# ---------------- 3. Decoder：在64x64/32x32两处插入GCN skip ----------------
class DecoderGCN(nn.Module):
    def __init__(self, base_decoder_attn):
        super().__init__()
        old = list(base_decoder_attn.conv_layers)
        # 索引对照(来自DecoderAttn): 0=conv_in, 1-3=bottleneck, 4-5=mul4(16x16),
        #   6=upsample(->32x32), 7=已有zero-init attn(32x32), 8-9=mul2(32x32->64ch),
        #   10=upsample(->64x64), 11-12=mul1(64x64->32ch), 13=upsample(->128x128),
        #   14=norm, 15=conv_out, 16=sigmoid
        self.conv_in    = old[0]
        self.bottleneck = nn.Sequential(*old[1:4])
        self.stage4 = nn.Sequential(*old[4:6])       # 16x16, 128ch
        self.up1    = old[6]                          # ->32x32
        self.attn32 = old[7]                          # 已有的zero-init attention(32x32)
        self.gcn_32 = GraphConvSkipBlock(128)          # 新增：32x32的GCN skip
        self.stage2 = nn.Sequential(*old[8:10])       # 32x32->64ch
        self.up2    = old[10]                          # ->64x64
        self.gcn_64 = GraphConvSkipBlock(64)            # 新增：64x64的GCN skip
        self.stage1 = nn.Sequential(*old[11:13])      # 64x64->32ch
        self.up3    = old[13]                          # ->128x128
        self.norm_out = old[14]
        self.conv_out  = old[15]
        self.sigmoid   = old[16]

    def forward(self, z, feat_64, feat_32):
        h = self.conv_in(z)
        h = self.bottleneck(h)
        h = self.stage4(h)
        h = self.up1(h)
        h = self.attn32(h)
        h = self.gcn_32(h, feat_32)          # <- 用encoder的32x32特征算邻接矩阵
        h = self.stage2(h)
        h = self.up2(h)
        h = self.gcn_64(h, feat_64)          # <- 用encoder的64x64特征算邻接矩阵
        h = self.stage1(h)
        h = self.up3(h)
        h = self.norm_out(h)
        h = self.conv_out(h)
        return self.sigmoid(h)


class BinaryAutoEncoderGCN(nn.Module):
    def __init__(self, base_ae_attn):
        super().__init__()
        self.encoder = EncoderGCN(base_ae_attn.encoder)
        self.decoder = DecoderGCN(base_ae_attn.decoder)
        self.latent_h = base_ae_attn.latent_h
        self.latent_w = base_ae_attn.latent_w
        self.D = base_ae_attn.D

    def encode_binary(self, x):
        with torch.no_grad():
            z, _, _ = self.encoder(x, hard=True)
        return z.long().view(z.shape[0], -1)

    def forward(self, x):
        z, feat_64, feat_32 = self.encoder(x, hard=True)
        x_rec = self.decoder(z, feat_64, feat_32)
        return x_rec, z


# ---------------- 4. 构建 + 确认权重来自ae_attn(热启动，GCN部分zero-init全新) ----------------
ae_gcn = BinaryAutoEncoderGCN(ae_attn).to(bld.DEVICE)

n_new = sum(p.numel() for n, p in ae_gcn.named_parameters() if 'gcn_' in n)
n_total = sum(p.numel() for p in ae_gcn.parameters())
print(f"ae_gcn 构建完成：总参数量 {n_total}，其中GCN新增模块 {n_new}（zero-init，起点=ae_attn完全等价）")

In [ ]:
# ==============================================================================
# 【旧版 GCN 重建与参数共享检查｜仅配合上一旧版定义】
# 重新加载 ae_attn，并用旧版 deepcopy GCN 做 data_ptr 检查。
# 新版流程会在最终 mAP 单元独立加载 ae_attn，因此正常运行时请跳过本单元。
# ==============================================================================

# 1. 重新从磁盘加载一份干净的ae_attn，恢复被污染的状态
ae_attn = BinaryAutoEncoderAttn().to(bld.DEVICE)
ae_attn.load_state_dict(
    torch.load('/content/drive/MyDrive/bld_purify_checkpoints/ae_attn_taskaware_v2.pt', map_location=bld.DEVICE)
)
ae_attn.eval()
print("ae_attn 已恢复为干净状态")
# 2. 用修复后(deepcopy版)的EncoderGCN/DecoderGCN重新构建ae_gcn
ae_gcn = BinaryAutoEncoderGCN(ae_attn).to(bld.DEVICE)
# 3. sanity check：这次参数应该不同了
p1 = next(ae_gcn.parameters()).flatten()[:5]
p2 = next(ae_attn.parameters()).flatten()[:5]
print("是否共享内存(应为False):", ae_gcn.encoder.stem.weight.data_ptr() == ae_attn.encoder.conv_layers[0].weight.data_ptr())

In [ ]:
# ==============================================================================
# 【旧版 GCN mAP｜已停用】
# 该代码依赖后面才定义的新版类，原始顺序无法可靠执行；正文被保存为字符串，不会运行。
# 请运行最长定义块之后的最后一个“新版 GCN mAP”单元。
# ==============================================================================

LEGACY_GCN_EVAL_CODE = r'''
import os
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ---------------- 彻底修复 criterion.hyp 的兼容性包装类 ----------------
class DictAttributeWrapper(dict):
    """同时支持 dict['key'] 和 dict.key 访问的字典包装类"""
    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(f"'DictAttributeWrapper' object has no attribute '{name}'")

    def __setattr__(self, name, value):
        self[name] = value

    def __delattr__(self, name):
        try:
            del self[name]
        except KeyError:
            raise AttributeError(f"'DictAttributeWrapper' object has no attribute '{name}'")

# 应用包装类修复 criterion.hyp
if hasattr(criterion, 'hyp') and isinstance(criterion.hyp, dict):
    criterion.hyp = DictAttributeWrapper(criterion.hyp)

# ==============================================================================
#  1. 初始化与权重加载
# ==============================================================================
levels = auto_detect_skip_levels(ae_attn, img_size=bld.IMG_SIZE)
ae_gcn_config = GCNSkipAEConfig()

ae_gcn = BinaryAutoEncoderGCN(ae_attn, ae_gcn_config, levels).to(bld.DEVICE)
ae_gcn.load_state_dict(
    torch.load('/content/drive/MyDrive/gcn_ae_checkpoints/gcn_ae_ckpt.pt', map_location=bld.DEVICE)
)
ae_gcn.eval()
print("gcn_ae 权重加载完成")

diff_model_v3 = bld.ReverseModel(D=ae_gcn.D).to(bld.DEVICE)
diff_model_v3_path = '/content/drive/MyDrive/bld_purify_checkpoints/bld_diffusion_v3.pt'
diff_model_v3.load_state_dict(torch.load(diff_model_v3_path, map_location=bld.DEVICE))
diff_model_v3.eval()
print("diff_model_v3 权重加载成功！")

T_STAR = 40
MODE = "oracle"
OUT_ROOT = f'/content/drive/MyDrive/dfire_adv_eval_gcn_pgd_{MODE}'

for split in ['clean', 'adv', 'purified']:
    os.makedirs(f'{OUT_ROOT}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUT_ROOT}/{split}/labels', exist_ok=True)

progress_path = f'{OUT_ROOT}/done.txt'
done = set(open(progress_path).read().split()) if os.path.exists(progress_path) else set()
print(f"已完成 {len(done)} 张图")
progress_f = open(progress_path, 'a')

betas, flip_prob = bld.get_schedule()

# ==============================================================================
#  2. 主循环：PGD攻击 -> GCN Skip 净化 -> 图像落盘
# ==============================================================================
for x, targets, paths in tqdm(det_loader, desc=f"gcn_pgd attack+purify [{MODE}]"):
    names = [os.path.basename(p) for p in paths]
    if all(n in done for n in names):
        continue

    x = x.to(bld.DEVICE)

    if targets['bboxes'].shape[0] == 0:
        x_adv = x.clone()
    else:
        targets_gpu = {k: v.to(bld.DEVICE) for k, v in targets.items()}
        x_adv = pgd_attack(attack_yolo.model, criterion, x, targets_gpu)

    x_adv_128   = F.interpolate(x_adv, size=bld.IMG_SIZE, mode='bilinear', align_corners=False)
    x_clean_128 = F.interpolate(x,     size=bld.IMG_SIZE, mode='bilinear', align_corners=False)

    with torch.no_grad():
        # A. 干净图直接经过 GCN AE：测量当前重建架构的检测上限。
        z_clean, feats_clean = ae_gcn.encoder(x_clean_128, hard=True)
        x_clean_rec_128 = ae_gcn.decoder(z_clean, feats_clean)

        # B. 攻击图直接经过 GCN AE，不使用 diffusion。
        z_adv, feats_adv = ae_gcn.encoder(x_adv_128, hard=True)
        z_adv_flat = z_adv.view(z_adv.shape[0], -1).long()
        if MODE == "realistic":
            feats_for_decode = feats_adv
        elif MODE == "oracle":
            feats_for_decode = feats_clean
        else:
            raise ValueError(f"未知 MODE: {MODE}")

        x_adv_rec_128 = ae_gcn.decoder(z_adv, feats_for_decode)

        # C. 同一个攻击latent经过 diffusion，再交给同一个 GCN decoder。
        t_vec = torch.full(
            (z_adv_flat.shape[0],), T_STAR - 1,
            device=bld.DEVICE, dtype=torch.long
        )
        z_noised = bld.q_sample(z_adv_flat, t_vec, flip_prob)
        z_rec = bld.reverse(
            diff_model_v3, z_noised, T_STAR, betas, flip_prob
        ).float().view(z_adv.shape)
        x_pur_128 = ae_gcn.decoder(z_rec, feats_for_decode)

    x_clean_rec = F.interpolate(x_clean_rec_128, size=384, mode='bilinear', align_corners=False)
    x_adv_rec = F.interpolate(x_adv_rec_128, size=384, mode='bilinear', align_corners=False)
    x_pur = F.interpolate(x_pur_128, size=384, mode='bilinear', align_corners=False)

    save_as_dfire_split(x,           paths, f'{OUT_ROOT}/original_clean')
    save_as_dfire_split(x_adv,       paths, f'{OUT_ROOT}/adv_original')
    save_as_dfire_split(x_clean_rec, paths, f'{OUT_ROOT}/clean_reconstructed')
    save_as_dfire_split(x_adv_rec,   paths, f'{OUT_ROOT}/adv_reconstructed')
    save_as_dfire_split(x_pur,       paths, f'{OUT_ROOT}/adv_diffusion_purified')

    for n in names:
        progress_f.write(n + '\n')
    progress_f.flush()
    done.update(names)

progress_f.close()
print(f"\n全部完成，共 {len(done)} 张图")

# ==============================================================================
#  3. mAP 评估
# ==============================================================================
print(f"\n=== gcn_ae + 真实PGD攻击 [{MODE} mode] mAP对比 ===")
results = {}
for name in ['clean', 'adv', 'purified']:
    map50, map5095 = eval_map(f'{OUT_ROOT}/{name}', img_size=384)
    results[name] = (map50, map5095)
    print(f"{name:10s}  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")

recovery = (results['purified'][0] - results['adv'][0]) / max(results['clean'][0] - results['adv'][0], 1e-8)
print(f"\n净化恢复比例 [{MODE}] = {recovery:.1%}")
'''
print("旧版 GCN mAP 单元已禁用；请运行最长定义块之后的最后一个 GCN mAP 单元。")

In [ ]:
# ==============================================================================
# 【新版 GCN 主定义 + 可选续训｜最终流程必须运行】
# 包含自动层级识别、skip量化、GCN模型、训练/评估函数及严格续训入口。
# 默认 RUN_GCN_TRAINING=False：只定义代码，不会重新训练。
# ==============================================================================

import json
import os
import shutil
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Tuple, Dict, Callable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import yaml
from PIL import Image
from tqdm import tqdm

# 假设 bld 已经在当前脚本/notebook 中 import 好
ResnetBlock = bld.ResnetBlock
AttnBlock   = bld.AttnBlock
Downsample  = bld.Downsample
Upsample    = bld.Upsample
Normalize   = bld.Normalize


# ==================== 1. 架构自动识别 ====================

def _out_channels(module) -> Optional[int]:
    """尝试拿到一个bld模块的输出通道数；识别不了(比如Normalize/Sigmoid)就返回None，
    表示"沿用之前记录的通道数"（这些模块不改变通道数）"""
    if isinstance(module, nn.Conv2d):
        return module.out_channels
    if isinstance(module, ResnetBlock):
        return module.conv2.out_channels
    if isinstance(module, AttnBlock):
        return module.proj.out_channels
    return None


def _inspect_stages(seq, resize_cls) -> List[Dict]:
    """
    把一个 nn.Sequential 按 Downsample/Upsample 出现的位置切成若干段，每段对应一个分辨率级别。
    返回 [{"slice": (start,end), "channels": c}, ...]
    slice 是 [start,end) 左闭右开，不包含触发分辨率变化的那个 Downsample/Upsample 模块本身。
    """
    stages, start, ch = [], 0, None
    modules = list(seq)
    for i, m in enumerate(modules):
        c = _out_channels(m)
        if c is not None:
            ch = c
        if isinstance(m, resize_cls):
            stages.append({"slice": (start, i), "channels": ch})
            start = i + 1
    stages.append({"slice": (start, len(modules)), "channels": ch})
    return stages


@dataclass
class SkipLevelSpec:
    name: str                    # 比如"64x64"，仅用于打印/按名字启用
    channels: int
    enc_slice: Tuple[int, int]   # encoder.conv_layers里该级别的[start,end)，end之后紧接着Downsample
    dec_insert_after: int        # decoder.conv_layers里，在这个索引对应模块forward之后插入GCN


def auto_detect_skip_levels(base_ae, img_size: Optional[int] = None) -> List[SkipLevelSpec]:
    """
    自动扫描 base_ae.encoder / base_ae.decoder 的 conv_layers，定位所有可以插GCN skip的中间
    分辨率级别，自动排除最外层和bottleneck层。对 bld.BinaryAutoEncoder 和 ae_attn 都适用。
    """
    enc_stages = _inspect_stages(base_ae.encoder.conv_layers, Downsample)
    dec_stages = _inspect_stages(base_ae.decoder.conv_layers, Upsample)
    n_down = len(enc_stages) - 1

    if n_down != len(dec_stages) - 1:
        raise ValueError(
            f"encoder下采样次数({n_down}) != decoder上采样次数({len(dec_stages) - 1})，"
            f"这个AE的encoder/decoder不是对称结构，无法自动配对skip层级"
        )

    dec_modules = list(base_ae.decoder.conv_layers)
    levels = []
    for i in range(1, n_down):          # 跳过 i=0(最外层) 和 i=n_down(bottleneck)
        dec_j = n_down - i
        enc_seg = enc_stages[i]
        # decoder在该分辨率级别的"入口通道数" = 前一段(更小分辨率)处理完的通道数
        # (Upsample本身不改变通道数，所以入口通道 = 上一段的输出通道数)
        dec_in_channels = dec_stages[dec_j - 1]["channels"]
        upsample_idx = dec_stages[dec_j]["slice"][0] - 1   # 触发这次分辨率提升的Upsample模块索引

        if enc_seg["channels"] != dec_in_channels:
            print(f"[跳过] level{i}: encoder通道{enc_seg['channels']} != "
                  f"decoder入口通道{dec_in_channels}，架构不对称，无法插入GCN skip")
            continue

        res = img_size // (2 ** i) if img_size else None
        name = f"{res}x{res}" if res else f"level{i}"

        next_module = dec_modules[upsample_idx + 1] if upsample_idx + 1 < len(dec_modules) else None
        if isinstance(next_module, AttnBlock):
            print(f"[提示] {name}: Upsample之后紧接着一个已有的AttnBlock，"
                  f"自动插入点在它之前。如果想让GCN插在这个AttnBlock之后，"
                  f"对返回结果做 spec.dec_insert_after += 1")

        levels.append(SkipLevelSpec(
            name=name,
            channels=enc_seg["channels"],
            enc_slice=enc_seg["slice"],
            dec_insert_after=upsample_idx,
        ))
    return levels


# ==================== 2. 配置类 ====================

@dataclass
class GCNBlockConfig:
    """GraphConvSkipBlock 内部超参数，所有启用的层级共享同一份配置"""
    zero_init: bool = True                 # True=起点严格等价于base_ae(做对照实验建议保持True)
    temperature: Optional[float] = None    # None -> 用 channels**-0.5
    attn_dropout: float = 0.0              # softmax(adj)之后的dropout，防止过拟合到训练集的图结构


@dataclass
class GCNSkipAEConfig:
    active_levels: Optional[List[str]] = None   # None=启用auto_detect找到的全部层级；
                                                  # 否则传层级名字列表，如["32x32"]做消融
    block_cfg: GCNBlockConfig = field(default_factory=GCNBlockConfig)
    freeze_base: bool = False   # True=冻结除gcn_blocks外的所有参数，只训练新增GCN模块

    quantize_skip: bool = True       # True=skip特征过SkipQuantizer量化成有限bit，而不是原始float
    skip_code_channels: int = 8       # 量化码的通道数(每个level共用这个设置)
    skip_code_spatial: int = 8        # 量化码的空间分辨率(每个level共用这个设置)

    def to_json(self, path: str):
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2, ensure_ascii=False)

    @classmethod
    def from_json(cls, path: str):
        with open(path) as f:
            d = json.load(f)
        d["block_cfg"] = GCNBlockConfig(**d["block_cfg"])
        return cls(**d)


# ==================== 3. GraphConvSkipBlock ====================

class GraphConvSkipBlock(nn.Module):
    """
    邻接矩阵由 encoder 特征算出 (self-attention式 QK, 无V)
    特征矩阵是 decoder 当前层的特征
    H' = A @ X @ W, zero-init残差接回(可关闭)
    """
    def __init__(self, channels: int, cfg: GCNBlockConfig):
        super().__init__()
        self.cfg = cfg
        self.temperature = cfg.temperature if cfg.temperature is not None else channels ** -0.5

        self.norm_enc = Normalize(channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.gcn_w = nn.Linear(channels, channels)
        self.proj = nn.Conv2d(channels, channels, 1)
        self.attn_dropout = nn.Dropout(cfg.attn_dropout) if cfg.attn_dropout > 0 else nn.Identity()

        if cfg.zero_init:
            nn.init.zeros_(self.proj.weight)
            nn.init.zeros_(self.proj.bias)

    def forward(self, x_dec, feat_enc):
        B, C, H, W = x_dec.shape
        h_enc = self.norm_enc(feat_enc)
        q = self.q(h_enc).reshape(B, C, -1).permute(0, 2, 1)       # (B,N,C)
        k = self.k(h_enc).reshape(B, C, -1)                         # (B,C,N)
        adj = torch.softmax(torch.bmm(q, k) * self.temperature, dim=-1)
        adj = self.attn_dropout(adj)

        x_flat = x_dec.reshape(B, C, -1).permute(0, 2, 1)           # (B,N,C)
        agg = torch.bmm(adj, x_flat)                                 # A @ X
        agg = self.gcn_w(agg)                                        # A @ X @ W
        agg = agg.permute(0, 2, 1).reshape(B, C, H, W)

        return x_dec + self.proj(agg)


class SkipQuantizer(nn.Module):
    """
    把encoder某一层级的原始float中间特征，压缩成一个固定长度的二进制码，
    再解码回一个和原始特征同样大小的"代理特征图"，供GraphConvSkipBlock的Q/K使用。

    这样skip路径携带的信息量从"未知精度的float，理论上bit预算无限"，
    变成"显式的 code_channels × code_spatial × code_spatial 个bit"，可以和主传输的
    2048bit加在一起，报告一个诚实的总比特预算，而不是含糊地说"额外用了一些float特征"。

    注意：这一步只是把skip路径的信息量限定住了。如果要更严格地评估"净化框架
    对这部分bit也有效"，还需要把这些bit也接入diffusion净化流程(比如拼到主z里
    一起加噪/净化，或者单独训一个净化器)，这是另一步，这里没有做。
    """
    def __init__(self, channels: int, code_channels: int = 8, code_spatial: int = 8):
        super().__init__()
        self.n_bits = code_channels * code_spatial * code_spatial

        self.pool = nn.AdaptiveAvgPool2d(code_spatial)
        self.encode_conv = nn.Conv2d(channels, code_channels, 1)
        self.decode_conv = nn.Conv2d(code_channels, channels, 1)

    def encode(self, feat_enc):
        """feat_enc -> 二进制码 (B, code_channels, code_spatial, code_spatial)，取值{0,1}"""
        h = self.pool(feat_enc)
        p = torch.sigmoid(self.encode_conv(h))
        hard = (p > 0.5).float()
        return p + (hard - p).detach()   # straight-through，和主z的二值化用同样的技巧

    def decode(self, code, spatial_size):
        """二进制码 -> 还原成spatial_size大小的代理特征图，供GraphConvSkipBlock使用。
        spatial_size在forward时从实际的feat_enc张量形状里读，不需要在构造时预先指定，
        这样不用依赖level名字里的分辨率数字去解析(更稳健，不用管命名格式)。"""
        h = F.interpolate(code, size=(spatial_size, spatial_size), mode='nearest')
        return self.decode_conv(h)

    def forward(self, feat_enc):
        code = self.encode(feat_enc)
        feat_proxy = self.decode(code, feat_enc.shape[-1])
        return feat_proxy, code


# ==================== 4. Encoder / Decoder 包装（动态插入，不再手写子模块） ====================

class EncoderGCN(nn.Module):
    """直接复用原始encoder的conv_layers，逐模块forward，在探测到的每个level边界处记录中间特征"""
    def __init__(self, base_encoder, levels: List[SkipLevelSpec]):
        super().__init__()
        self.base_encoder = base_encoder
        self.levels = levels
        # 每个level对应"该分辨率处理完、Downsample之前"的模块索引
        self._hook_at: Dict[int, str] = {lv.enc_slice[1] - 1: lv.name for lv in levels}

    def forward(self, x, hard=True):
        feats = {}
        h = x
        for i, m in enumerate(self.base_encoder.conv_layers):
            h = m(h)
            if i in self._hook_at:
                feats[self._hook_at[i]] = h
        logits = h

        if not hard:
            z = torch.sigmoid(logits)
        else:
            z_soft = torch.sigmoid(logits)
            z_hard = (z_soft > 0.5).float()
            z = z_soft + (z_hard - z_soft).detach()
        return z, feats


class DecoderGCN(nn.Module):
    """直接复用原始decoder的conv_layers，逐模块forward，在启用的level对应索引后插入GCN block"""
    def __init__(self, base_decoder, levels: List[SkipLevelSpec], config: GCNSkipAEConfig):
        super().__init__()
        self.base_decoder = base_decoder
        self.config = config
        active = set(config.active_levels) if config.active_levels is not None else None

        self.gcn_blocks = nn.ModuleDict()
        self.skip_quantizers = nn.ModuleDict()   # 只有 quantize_skip=True 时才会有内容
        self._insert_after: Dict[int, str] = {}
        for lv in levels:
            if active is not None and lv.name not in active:
                continue
            self.gcn_blocks[lv.name] = GraphConvSkipBlock(lv.channels, config.block_cfg)
            if config.quantize_skip:
                self.skip_quantizers[lv.name] = SkipQuantizer(
                    lv.channels,
                    code_channels=config.skip_code_channels,
                    code_spatial=config.skip_code_spatial,
                )
            self._insert_after[lv.dec_insert_after] = lv.name

    def forward(self, z, enc_feats: Dict[str, torch.Tensor]):
        h = z
        for i, m in enumerate(self.base_decoder.conv_layers):
            h = m(h)
            if i in self._insert_after:
                name = self._insert_after[i]
                feat = enc_feats[name]
                if name in self.skip_quantizers:
                    feat, _code = self.skip_quantizers[name](feat)   # 量化+解码成代理特征
                h = self.gcn_blocks[name](h, feat)
        return h

    def skip_bit_budget(self) -> int:
        """这个decoder里所有量化skip层级加起来一共用了多少bit(quantize_skip=False时返回0)"""
        return sum(q.n_bits for q in self.skip_quantizers.values())


# ==================== 5. 完整模型 ====================

class BinaryAutoEncoderGCN(nn.Module):
    def __init__(self, base_ae, config: Optional[GCNSkipAEConfig] = None,
                 levels: Optional[List[SkipLevelSpec]] = None,
                 img_size: Optional[int] = None):
        super().__init__()
        self.config = config or GCNSkipAEConfig()
        img_size = img_size or getattr(bld, "IMG_SIZE", None)
        self.levels = levels if levels is not None else auto_detect_skip_levels(base_ae, img_size)

        self.encoder = EncoderGCN(base_ae.encoder, self.levels)
        self.decoder = DecoderGCN(base_ae.decoder, self.levels, self.config)
        self.latent_h = base_ae.latent_h
        self.latent_w = base_ae.latent_w
        self.D = base_ae.D

        if self.config.freeze_base:
            self._freeze_non_gcn_params()

    def _freeze_non_gcn_params(self):
        n_frozen = 0
        for name, p in self.named_parameters():
            if "gcn_blocks" not in name:
                p.requires_grad = False
                n_frozen += p.numel()
        print(f"[freeze_base=True] 已冻结 {n_frozen:,} 个非GCN参数，仅GCN模块可训练")

    def encode_binary(self, x):
        with torch.no_grad():
            z, _ = self.encoder(x, hard=True)
        return z.long().view(z.shape[0], -1)

    def encode_with_feats(self, x):
        """和encode_binary类似，但同时返回GCN decoder需要的encoder中间特征。
        净化评估这类场景需要单独decode一个被攻击/净化过的z(而不是走完整的forward(x))，
        这时候decode必须配上对应的encoder特征，所以拆出这个方法。"""
        with torch.no_grad():
            z, feats = self.encoder(x, hard=True)
        return z.long().view(z.shape[0], -1), feats

    def decode(self, z_flat, enc_feats):
        """解码一个(可能是被攻击/净化过的)binary latent。
        注意：enc_feats必须来自某张干净图片的encode_with_feats，因为GCN的邻接矩阵
        依赖它。也就是说这个架构没法做到"只靠传输的bit就能解码"——接收端必须额外
        持有一份对应的encoder特征。这在离线评估(z和干净图配对已知)里没问题，但如果
        以后要做真实的"压缩后传输、接收端只有bit"部署场景，这是需要另外处理的架构限制，
        不是这次净化评估要解决的问题，先在这里提醒一下。"""
        B = z_flat.shape[0]
        n_latent_ch = self.D // (self.latent_h * self.latent_w)
        z = z_flat.view(B, n_latent_ch, self.latent_h, self.latent_w).float()
        return self.decoder(z, enc_feats)

    def forward(self, x):
        z, feats = self.encoder(x, hard=True)
        x_rec = self.decoder(z, feats)
        return x_rec, z

    def total_bit_budget(self) -> int:
        """主传输bit(self.D) + 所有量化skip层级的bit，是这个模型诚实的总比特预算。
        quantize_skip=False时，skip_bit_budget()恒为0，total就等于self.D(和v4一致)。"""
        return self.D + self.decoder.skip_bit_budget()

    def summary(self):
        n_total = sum(p.numel() for p in self.parameters())
        n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        n_gcn = sum(p.numel() for n, p in self.named_parameters() if "gcn_blocks" in n)
        skip_bits = self.decoder.skip_bit_budget()
        print(f"检测到的可用skip层级: {[lv.name for lv in self.levels]}")
        print(f"实际启用: {list(self.decoder.gcn_blocks.keys())}")
        print(f"总参数量: {n_total:,}  可训练: {n_trainable:,}  GCN新增: {n_gcn:,}")
        print(f"zero_init={self.config.block_cfg.zero_init}  temperature={self.config.block_cfg.temperature}  "
              f"attn_dropout={self.config.block_cfg.attn_dropout}  freeze_base={self.config.freeze_base}")
        if self.config.quantize_skip:
            print(f"比特预算: 主传输 {self.D} bit + skip量化 {skip_bits} bit "
                  f"= 总计 {self.total_bit_budget()} bit "
                  f"(相比纯2048bit的v4，多了 {skip_bits} bit 的显式辅助信息)")
        else:
            print(f"比特预算: 主传输 {self.D} bit (skip特征未量化，仍是未计量的float，"
                  f"不能报告为固定bit数——如果要一个诚实的总预算数字，"
                  f"把GCNSkipAEConfig(quantize_skip=True)打开)")


# ==================== 6. 工厂函数 ====================

def build_gcn_ae(base_ae, config: Optional[GCNSkipAEConfig] = None,
                  levels: Optional[List[SkipLevelSpec]] = None,
                  img_size: Optional[int] = None) -> BinaryAutoEncoderGCN:
    model = BinaryAutoEncoderGCN(base_ae, config, levels, img_size).to(bld.DEVICE)
    model.summary()
    return model


# ==================== 7. 训练循环（两阶段：ae_only -> 接det_loss） ====================

@dataclass
class TrainConfig:
    epochs_ae_only: int = 10        # 阶段1：只用MSE+binary正则，先验证GCN架构本身有效
    epochs_with_det: int = 10       # 阶段2：接入det_loss联合训练；设0则只跑阶段1
    lr: float = 1e-4
    det_loss_weight: float = 0.1    # λ，对应你CRIS里det_loss的权重
    det_warm_start_epochs: int = 3  # det_loss权重从0线性warm-start到det_loss_weight所用的epoch数
    bin_reg_weight: float = 0.01    # 和bld.py的train_autoencoder保持一致
    ckpt_path: str = "gcn_ae_ckpt.pt"        # 本地路径(快，每次保存都写这里)
    drive_backup_path: Optional[str] = None  # 如果设了，每次保存最优checkpoint时额外同步一份到这里，
                                              # 并验证复制成功(存在+大小一致)。强烈建议在Colab里设成
                                              # Drive路径，比如 '/content/drive/MyDrive/xxx/gcn_ae_ckpt.pt'


def save_ckpt_safely(model, ckpt_path: str, drive_backup_path: Optional[str] = None):
    """
    先存本地(快、不受Drive同步延迟影响)，如果指定了drive_backup_path，
    再显式复制过去并验证(文件存在 + 大小一致)，验证失败会抛错而不是静默失败——
    这样"看起来存了但其实没同步"这种情况会在训练时就报错，而不是等断连之后才发现。
    """
    torch.save(model.state_dict(), ckpt_path)
    if drive_backup_path is None:
        return
    os.makedirs(os.path.dirname(drive_backup_path), exist_ok=True)
    shutil.copy(ckpt_path, drive_backup_path)
    local_size = os.path.getsize(ckpt_path)
    drive_size = os.path.getsize(drive_backup_path)
    if drive_size != local_size:
        raise RuntimeError(
            f"Drive备份校验失败: 本地{local_size}字节 vs Drive{drive_size}字节，"
            f"可能是Drive同步没跟上，请重新保存或检查Drive挂载状态"
        )
    print(f"    ✓ 已同步到Drive: {drive_backup_path} ({drive_size / 1e6:.1f} MB)")


def evaluate_gcn_ae(model, loader, n_batches=5):
    """和bld.evaluate_ae同样的PSNR评估方式，适配GCN模型forward返回(x_rec, z)的签名"""
    model.eval()
    psnrs = []
    with torch.no_grad():
        for i, (x, _) in enumerate(loader):
            if i >= n_batches:
                break
            x = x.to(bld.DEVICE)
            x_rec, _ = model(x)
            mse = F.mse_loss(x_rec, x, reduction='none').mean(dim=(1, 2, 3))
            psnr = (-10 * torch.log10(mse + 1e-8)).mean().item()
            psnrs.append(psnr)
    return float(np.mean(psnrs))


def train_gcn_ae(model, train_loader, test_loader, train_cfg: TrainConfig,
                  det_loss_fn: Optional[Callable[[torch.Tensor, torch.Tensor], torch.Tensor]] = None):
    """
    两阶段训练：
      阶段1 (epochs_ae_only): 只用 MSE + binary正则，先确认GCN架构本身在ae_only层面有效，
                               对应你一贯的"先ae_only验证，再接完整pipeline"的习惯。
      阶段2 (epochs_with_det): 如果传入了det_loss_fn，在阶段1基础上继续训练，
                               det_loss权重从0线性warm-start到train_cfg.det_loss_weight。
    如果 det_loss_fn=None，只跑阶段1(纯重建，不接检测loss)。
    如果 model 是用 freeze_base=True 构建的，optimizer只会拿到requires_grad=True的
    那部分参数(也就是GCN新增模块)，不需要手动过滤。

    det_loss_fn 签名: det_loss_fn(x_rec, x_orig) -> scalar tensor
    (比如你CRIS pipeline里接YOLOv8n算检测特征loss的那个函数，直接传进来即可)
    """
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=train_cfg.lr)
    total_epochs = train_cfg.epochs_ae_only + train_cfg.epochs_with_det
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(total_epochs, 1))

    best_psnr = 0.0
    epoch_idx = 0

    def run_epoch(use_det_loss, det_weight):
        nonlocal epoch_idx
        model.train()
        total_loss, n = 0.0, 0
        for x, _ in tqdm(train_loader, desc=f"epoch {epoch_idx + 1}/{total_epochs}", leave=False):
            x = x.to(bld.DEVICE)
            x_rec, z = model(x)

            mse = F.mse_loss(x_rec, x)
            z_soft = z.float()
            bin_reg = -(z_soft * torch.log(z_soft + 1e-6) +
                        (1 - z_soft) * torch.log(1 - z_soft + 1e-6)).mean()
            loss = mse + train_cfg.bin_reg_weight * bin_reg

            if use_det_loss and det_loss_fn is not None:
                loss = loss + det_weight * det_loss_fn(x_rec, x)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()

            total_loss += loss.item() * x.shape[0]
            n += x.shape[0]

        scheduler.step()
        avg_loss = total_loss / max(n, 1)
        psnr = evaluate_gcn_ae(model, test_loader)
        det_str = f"  det_weight={det_weight:.4f}" if use_det_loss else ""
        print(f"  epoch {epoch_idx + 1}/{total_epochs}  loss={avg_loss:.4f}  PSNR={psnr:.2f}dB{det_str}")
        epoch_idx += 1
        return psnr

    print(f"=== 阶段1: ae_only ({train_cfg.epochs_ae_only} epochs) ===")
    for _ in range(train_cfg.epochs_ae_only):
        psnr = run_epoch(use_det_loss=False, det_weight=0.0)
        if psnr > best_psnr:
            best_psnr = psnr
            save_ckpt_safely(model, train_cfg.ckpt_path, train_cfg.drive_backup_path)
            print(f"    √ checkpoint saved (best PSNR={best_psnr:.2f}dB)")

    if train_cfg.epochs_with_det > 0 and det_loss_fn is not None:
        print(f"=== 阶段2: 接入det_loss，warm-start {train_cfg.det_warm_start_epochs} epochs ===")
        for e in range(train_cfg.epochs_with_det):
            warm_frac = min(1.0, (e + 1) / max(train_cfg.det_warm_start_epochs, 1))
            det_weight = train_cfg.det_loss_weight * warm_frac
            psnr = run_epoch(use_det_loss=True, det_weight=det_weight)
            if psnr > best_psnr:
                best_psnr = psnr
                save_ckpt_safely(model, train_cfg.ckpt_path, train_cfg.drive_backup_path)
                print(f"    √ checkpoint saved (best PSNR={best_psnr:.2f}dB)")
    elif train_cfg.epochs_with_det > 0 and det_loss_fn is None:
        print("[提示] epochs_with_det>0但没有传入det_loss_fn，跳过阶段2")

    print(f"训练完成。最佳PSNR: {best_psnr:.2f}dB，checkpoint保存在 {train_cfg.ckpt_path}")
    model.load_state_dict(torch.load(train_cfg.ckpt_path, map_location=bld.DEVICE))
    return model


# ==================== 8. mAP评估（对接现有的YOLOv8n eval） ====================

def save_as_dfire_split(x_batch, paths, save_dir):
    """把一个batch的重建图像存成D-Fire格式的images/+labels/，label从原图对应的.txt复制过来。
    (原样保留，未改动)"""
    os.makedirs(f"{save_dir}/images", exist_ok=True)
    os.makedirs(f"{save_dir}/labels", exist_ok=True)
    for img, p in zip(x_batch, paths):
        name = os.path.basename(p).rsplit('.', 1)[0]
        T.ToPILImage()(img.cpu()).save(f"{save_dir}/images/{name}.jpg")
        lp = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        if os.path.exists(lp):
            shutil.copy(lp, f"{save_dir}/labels/{name}.txt")


def eval_map(save_dir, img_size=384, detector=None):
    """(原样保留，只是把detector从隐式全局变量改成显式参数，避免这个脚本单独运行时
    依赖notebook里才有的全局状态；不传的话会尝试从全局作用域找)"""
    if detector is None:
        detector = globals().get("detector")
        assert detector is not None, "需要传入detector参数，或在全局作用域定义好detector(YOLOv8n模型)"
    yaml_path = f"{save_dir}/tmp.yaml"
    with open(yaml_path, 'w') as f:
        yaml.dump({'path': save_dir, 'train': 'images', 'val': 'images',
                   'names': {0: 'fire', 1: 'smoke'}}, f)
    m = detector.val(data=yaml_path, imgsz=img_size, verbose=False)
    return m.box.map50, m.box.map


def evaluate_gcn_ae_map(model, dataset, detector, save_dir="/tmp/gcn_ae_map_eval",
                         batch_size=16, n_samples=None, img_size=384):
    """
    对model重建后的图像跑一次真实mAP评估，复用上面的save_as_dfire_split + eval_map。

    dataset 需要有 .paths(图片路径列表) 和 .tf(预处理transform)，也就是直接传
    bld.DFireDataset 的实例，不是DataLoader —— 因为mAP评估需要知道每张图对应哪个
    label文件，这个对应关系只在 dataset.paths 里有，DataLoader batch出来的tensor
    丢失了这个信息，所以这里按 dataset.paths 手动分batch，不经过DataLoader。

    n_samples: 只评估前n_samples张图。mAP评估比PSNR慢很多(要存图+跑一次完整YOLO val)，
    调试阶段建议先设小一点，比如n_samples=200，确认流程通了再上全量测试集。
    """
    model.eval()
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)   # 清空，避免和上一次评估的残留图混在一起

    paths = dataset.paths if n_samples is None else dataset.paths[:n_samples]

    with torch.no_grad():
        for start in range(0, len(paths), batch_size):
            batch_paths = paths[start:start + batch_size]
            x_batch = torch.stack(
                [dataset.tf(Image.open(p).convert("RGB")) for p in batch_paths]
            ).to(bld.DEVICE)
            x_rec, _ = model(x_batch)
            save_as_dfire_split(x_rec, batch_paths, save_dir)

    map50, map5095 = eval_map(save_dir, img_size=img_size, detector=detector)
    print(f"[mAP eval] map50={map50:.4f}  map50-95={map5095:.4f}  "
          f"(n={len(paths)}, save_dir={save_dir})")
    return map50, map5095


# ==================== 9. mAP口径的净化评估（攻击→净化→解码→检测） ====================

def evaluate_purification_map(
        diff_model, ae, dataset, detector, betas, flip_prob,
        flip_rates=None, t_stars=None,
        n_samples=200, batch_size=16, img_size=384,
        save_dir_root="/content/eval_purify_map"):
    """
    mAP口径的净化评估，和bld.evaluate用同一套攻击模拟方式(binary latent空间的bit-flip腐蚀)，
    只是把bit accuracy换成真实mAP，方便和你v1-v4的purification recovery ratio对齐比较。

    如果你要的是真实PGD攻击(pixel空间对YOLOv8n loss求梯度)而不是这里的bit-flip，这个函数
    不适用，需要另外写一版——把你CRIS pipeline里那部分PGD代码发我，我可以在攻击这一步替换掉，
    net化→解码→检测的部分可以直接复用。

    重要架构提醒：因为GCN skip的邻接矩阵依赖encoder中间特征，decode一个被攻击/净化过的z时，
    这里用的是"同一张干净图片自己的encoder特征"配对解码——不是模拟"接收端只有bit、没有原图"
    的真实部署场景。这对离线评估"净化能不能把mAP拉回来"是合理的(z和干净图本来就是配对已知的)，
    但如果你以后要写"这套架构能不能真的部署"这类结论，这个前提需要额外说明。

    n_samples: 网格是 (1 + len(flip_rates)*(1+len(t_stars))) 次完整YOLO val，
    默认flip_rates/t_stars(4×5)算下来是25次，比较慢。调试阶段强烈建议先传小一点的
    flip_rates=[0.1]、t_stars=[10,40]、n_samples=50 跑通流程，确认没问题再上完整网格。

    返回: {"clean": (map50,map), "attacked_fr0.1": (map50,map),
           "purified_fr0.1_t10": (map50,map), ...}
    """
    diff_model.eval()
    ae.eval()
    flip_rates = flip_rates if flip_rates is not None else bld.FLIP_RATES
    t_stars = t_stars if t_stars is not None else bld.T_STARS
    paths = dataset.paths[:n_samples]

    dirs = {"clean": f"{save_dir_root}/clean"}
    for fr in flip_rates:
        dirs[f"attacked_fr{fr}"] = f"{save_dir_root}/attacked_fr{fr}"
        for t in t_stars:
            dirs[f"purified_fr{fr}_t{t}"] = f"{save_dir_root}/purified_fr{fr}_t{t}"

    for d in dirs.values():
        if os.path.exists(d):
            shutil.rmtree(d)

    print(f"共 {len(dirs)} 组配置 × {len(paths)} 张图，开始生成重建图像...")
    with torch.no_grad():
        for start in range(0, len(paths), batch_size):
            batch_paths = paths[start:start + batch_size]
            x = torch.stack(
                [dataset.tf(Image.open(p).convert("RGB")) for p in batch_paths]
            ).to(bld.DEVICE)

            z_clean, feats = ae.encode_with_feats(x)

            x_rec_clean = ae.decode(z_clean, feats)
            save_as_dfire_split(x_rec_clean, batch_paths, dirs["clean"])

            for fr in flip_rates:
                z_adv = ((z_clean.float() +
                          torch.bernoulli(torch.full_like(z_clean.float(), fr))) % 2).long()

                x_rec_adv = ae.decode(z_adv, feats)
                save_as_dfire_split(x_rec_adv, batch_paths, dirs[f"attacked_fr{fr}"])

                for t_star in t_stars:
                    t_vec = torch.full((z_adv.shape[0],), t_star - 1,
                                        device=bld.DEVICE, dtype=torch.long)
                    z_noised = bld.q_sample(z_adv, t_vec, flip_prob)
                    z_rec = bld.reverse(diff_model, z_noised, t_star, betas, flip_prob)
                    x_rec_purified = ae.decode(z_rec, feats)
                    save_as_dfire_split(x_rec_purified, batch_paths,
                                         dirs[f"purified_fr{fr}_t{t_star}"])

    print("图像生成完毕，开始跑YOLO验证...")
    results = {}
    results["clean"] = eval_map(dirs["clean"], img_size=img_size, detector=detector)
    print(f"  [clean]  map50={results['clean'][0]:.4f}  map={results['clean'][1]:.4f}")

    for fr in flip_rates:
        key = f"attacked_fr{fr}"
        results[key] = eval_map(dirs[key], img_size=img_size, detector=detector)
        m50, m = results[key]
        print(f"  [attacked fr={fr:.0%}]  map50={m50:.4f}  map={m:.4f}")

        for t_star in t_stars:
            key = f"purified_fr{fr}_t{t_star}"
            results[key] = eval_map(dirs[key], img_size=img_size, detector=detector)
            m50, m = results[key]
            print(f"    [purified fr={fr:.0%} t*={t_star}]  map50={m50:.4f}  map={m:.4f}")

    return results


# ==================== 可选训练入口 ====================
# Notebook 中 __name__ 通常也是 '__main__'，所以不能用它控制是否训练。
# 已有 GCN checkpoint 做 mAP 时保持 False；只有明确需要重新训练时才改为 True。
RUN_GCN_TRAINING = False
RESUME_GCN_CKPT = '/content/drive/MyDrive/gcn_ae_checkpoints/gcn_ae_ckpt.pt'
TRAIN_LOCAL_CKPT = '/content/gcn_ae_ckpt.pt'
TRAIN_DRIVE_BACKUP = '/content/drive/MyDrive/gcn_ae_checkpoints/gcn_ae_ckpt.pt'
if RUN_GCN_TRAINING:
    # 每次训练都从磁盘重建独立 base AE，避免被前面旧模型或共享参数污染。
    ae_attn_train = BinaryAutoEncoderAttn().to(bld.DEVICE)
    ae_attn_train.load_state_dict(
        torch.load(
            '/content/drive/MyDrive/bld_purify_checkpoints/ae_attn_taskaware_v2.pt',
            map_location=bld.DEVICE,
        ),
        strict=True,
    )

    levels = auto_detect_skip_levels(ae_attn_train, img_size=bld.IMG_SIZE)
    for lv in levels:
        print(lv)

    train_gcn_config = GCNSkipAEConfig()
    ae_gcn = build_gcn_ae(ae_attn_train, train_gcn_config, levels=levels)

    # Drive 上已有兼容 checkpoint 时从它继续；不存在时才从 base AE + zero-init GCN 开始。
    if RESUME_GCN_CKPT and os.path.exists(RESUME_GCN_CKPT):
        resume_raw = torch.load(RESUME_GCN_CKPT, map_location=bld.DEVICE)
        if isinstance(resume_raw, dict) and 'model_state_dict' in resume_raw:
            resume_state = resume_raw['model_state_dict']
        elif isinstance(resume_raw, dict) and 'state_dict' in resume_raw:
            resume_state = resume_raw['state_dict']
        else:
            resume_state = resume_raw
        ae_gcn.load_state_dict(resume_state, strict=True)
        print('从已有 GCN checkpoint 继续训练:', RESUME_GCN_CKPT)
    else:
        print('未找到兼容的续训 checkpoint，将从新建 GCN 开始训练')

    train_loader, test_loader = bld.get_loaders()
    train_cfg = TrainConfig(
        epochs_ae_only=10,
        epochs_with_det=0,
        ckpt_path=TRAIN_LOCAL_CKPT,
        drive_backup_path=TRAIN_DRIVE_BACKUP,
    )
    ae_gcn = train_gcn_ae(ae_gcn, train_loader, test_loader, train_cfg, det_loss_fn=None)

    # 若要接检测损失，必须先实现真实可微的 det_loss_fn，再把 epochs_with_det 设为正数。
    # def my_det_loss_fn(x_rec, x_orig):
    #     ... # 例如用YOLOv8n算检测特征loss
    #     return loss
    # train_cfg2 = TrainConfig(epochs_ae_only=0, epochs_with_det=10,
    #                           det_loss_weight=0.1, det_warm_start_epochs=3)
    # ae_gcn = train_gcn_ae(ae_gcn, train_loader, test_loader, train_cfg2, det_loss_fn=my_det_loss_fn)

In [ ]:
# ==============================================================================
# 【新版 GCN mAP｜加载已有权重｜最终检测单元】
# 流程：严格加载 GCN → 检查活动层级 → 检查 proj 非全零 → PGD → diffusion → GCN解码 → mAP。
# 还会用 forward hook 验证每个 GCN block 确实被调用；本单元不训练任何模型。
# ==============================================================================

import glob
import os
import zlib
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

# ---------------- 彻底修复 criterion.hyp 的兼容性包装类 ----------------
class DictAttributeWrapper(dict):
    """同时支持 dict['key'] 和 dict.key 访问的字典包装类"""
    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(f"'DictAttributeWrapper' object has no attribute '{name}'")

    def __setattr__(self, name, value):
        self[name] = value

    def __delattr__(self, name):
        try:
            del self[name]
        except KeyError:
            raise AttributeError(f"'DictAttributeWrapper' object has no attribute '{name}'")

# 应用包装类修复 criterion.hyp
if hasattr(criterion, 'hyp') and isinstance(criterion.hyp, dict):
    criterion.hyp = DictAttributeWrapper(criterion.hyp)

# ==============================================================================
#  1. 独立重建模型，并严格加载已经训练好的 GCN checkpoint
# ==============================================================================
AE_ATTN_CKPT = '/content/drive/MyDrive/bld_purify_checkpoints/ae_attn_taskaware_v2.pt'
GCN_CKPT = '/content/drive/MyDrive/gcn_ae_checkpoints/gcn_ae_ckpt.pt'
DIFF_CKPT = '/content/drive/MyDrive/bld_purify_checkpoints/bld_diffusion_v3.pt'

for required_path in [AE_ATTN_CKPT, GCN_CKPT, DIFF_CKPT]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f'找不到权重文件: {required_path}')

# 不复用当前内存里的 ae_attn，防止它曾被旧训练代码原地修改。
ae_attn_eval = BinaryAutoEncoderAttn().to(bld.DEVICE)
ae_attn_eval.load_state_dict(
    torch.load(AE_ATTN_CKPT, map_location=bld.DEVICE), strict=True
)
ae_attn_eval.eval()

levels = auto_detect_skip_levels(ae_attn_eval, img_size=bld.IMG_SIZE)

# 下面必须和训练该 checkpoint 时的配置一致。当前值对应最长块的默认配置。
gcn_config = GCNSkipAEConfig(
    active_levels=None,
    block_cfg=GCNBlockConfig(
        zero_init=True, temperature=None, attn_dropout=0.0
    ),
    freeze_base=False,
    quantize_skip=True,
    skip_code_channels=8,
    skip_code_spatial=8,
)

ae_gcn = BinaryAutoEncoderGCN(
    ae_attn_eval, gcn_config, levels
).to(bld.DEVICE)

raw_ckpt = torch.load(GCN_CKPT, map_location=bld.DEVICE)
if isinstance(raw_ckpt, dict) and 'model_state_dict' in raw_ckpt:
    gcn_state_dict = raw_ckpt['model_state_dict']
elif isinstance(raw_ckpt, dict) and 'state_dict' in raw_ckpt:
    gcn_state_dict = raw_ckpt['state_dict']
else:
    gcn_state_dict = raw_ckpt

# strict=True 是关键：不允许把不兼容的旧结构权重悄悄当成加载成功。
ae_gcn.load_state_dict(gcn_state_dict, strict=True)
ae_gcn.eval()

# ---------------- GCN有效性预检：层级存在 + proj不为全0 ----------------
# proj仍全0时，zero-init残差不会输出任何GCN增量，数值上仍等价于baseline。
active_gcn_levels = list(ae_gcn.decoder.gcn_blocks.keys())
if not active_gcn_levels:
    raise RuntimeError('GCN 模型构建后没有任何活动层级；active_levels 不能是 []')
print('已严格加载 GCN checkpoint:', GCN_CKPT)
print('实际启用的 GCN 层级:', active_gcn_levels)
gcn_proj_norms = {}
for level_name, block in ae_gcn.decoder.gcn_blocks.items():
    gcn_proj_norms[level_name] = block.proj.weight.detach().norm().item()
    print(f'  {level_name}: proj.weight norm={gcn_proj_norms[level_name]:.6f}')
if not any(v > 1e-12 for v in gcn_proj_norms.values()):
    raise RuntimeError(
        '所有 GCN proj 权重仍为零：该 checkpoint 在数值上仍等价于 baseline，'
        '请确认加载的是训练完成后的 GCN 权重。'
    )

diff_model_v3 = bld.ReverseModel(D=ae_gcn.D).to(bld.DEVICE)
diff_model_v3.load_state_dict(
    torch.load(DIFF_CKPT, map_location=bld.DEVICE), strict=True
)
diff_model_v3.eval()
print('diff_model_v3 权重加载成功:', DIFF_CKPT)

T_STAR = 40  # 扫描时可改为 5/10/20/40；每个值会写入独立目录
MODE = 'realistic'  # 真实口径：GCN skip 特征来自被攻击图像，不使用干净图 oracle
ATTACK_SEED = 20260822  # 保证不同 T_STAR 使用相同的 PGD 随机起点
EXPERIMENT_SCHEMA = 'five_way_ablation_v1'

# checkpoint、模式、T_STAR、攻击种子都写入目录；修改任一实验条件都不会复用旧图片。
ckpt_stamp = f'{os.path.getsize(GCN_CKPT)}_{int(os.path.getmtime(GCN_CKPT))}'
OUT_ROOT = (
    f'/content/drive/MyDrive/dfire_adv_eval_gcn_{ckpt_stamp}'
    f'_pgd_{MODE}_t{T_STAR}_seed{ATTACK_SEED}_{EXPERIMENT_SCHEMA}'
)
OUTPUT_SPLITS = [
    'original_clean',
    'adv_original',
    'clean_reconstructed',
    'adv_reconstructed',
    'adv_diffusion_purified',
]

# 明确重建完整 test split。当前 D-Fire 目录应有 4306 张；数量不符时立即停止。
EVAL_SPLIT = 'test'
EXPECTED_IMAGE_COUNT = 4306
EVAL_BATCH_SIZE = 8
gcn_eval_ds = DFireDetDataset(bld.DFIRE_ROOT, split=EVAL_SPLIT, size=384)
if len(gcn_eval_ds) != EXPECTED_IMAGE_COUNT:
    raise RuntimeError(
        f'{EVAL_SPLIT} split 实际找到 {len(gcn_eval_ds)} 张，'
        f'但期望 {EXPECTED_IMAGE_COUNT} 张；请检查 DFIRE_ROOT 和数据集解压目录。'
    )
gcn_eval_loader = DataLoader(
    gcn_eval_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=det_collate
)
eval_names = {os.path.basename(p) for p in gcn_eval_ds.img_paths}
if len(eval_names) != EXPECTED_IMAGE_COUNT:
    raise RuntimeError('test split 中存在重复文件名，无法安全使用 done.txt 断点续跑')
print(f'本次目标：重建 {EVAL_SPLIT} split 全部 {EXPECTED_IMAGE_COUNT} 张图')

for split in OUTPUT_SPLITS:
    os.makedirs(f'{OUT_ROOT}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUT_ROOT}/{split}/labels', exist_ok=True)

progress_path = f'{OUT_ROOT}/done.txt'
done_raw = set(open(progress_path).read().split()) if os.path.exists(progress_path) else set()
# done.txt 中只有在全部五个输出目录都实际存在图片时才算完成，避免残缺断点被误跳过。
done = {
    name for name in done_raw
    if name in eval_names and all(
        os.path.exists(f'{OUT_ROOT}/{split}/images/{name}')
        for split in OUTPUT_SPLITS
    )
}
remaining_count = EXPECTED_IMAGE_COUNT - len(done)
print(
    f'断点续跑状态：已有 {len(done)}/{EXPECTED_IMAGE_COUNT} 张，'
    f'本次还需生成 {remaining_count} 张'
)
progress_f = open(progress_path, 'a')

betas, flip_prob = bld.get_schedule()

# ---------------- GCN运行验证：统计每个block的实际forward次数 ----------------
# forward hook 会记录每个 GCN block 的实际调用次数，作为确实接入 GCN 的运行证据。
gcn_hits = {name: 0 for name in active_gcn_levels}
gcn_hook_handles = []
processed_batches = 0

def make_gcn_hook(name):
    def hook(module, inputs, output):
        gcn_hits[name] += 1
    return hook

for level_name, block in ae_gcn.decoder.gcn_blocks.items():
    gcn_hook_handles.append(
        block.register_forward_hook(make_gcn_hook(level_name))
    )

# ==============================================================================
#  2. 主循环：PGD攻击 -> diffusion净化 -> GCN Skip解码 -> 图像落盘
# ==============================================================================
for x, targets, paths in tqdm(gcn_eval_loader, desc=f"gcn_pgd attack+purify [{MODE}]"):
    names = [os.path.basename(p) for p in paths]
    if all(n in done for n in names):
        continue

    x = x.to(bld.DEVICE)

    # 按batch文件名生成稳定种子：断点续跑以及不同T_STAR都会得到完全相同的PGD随机初始化。
    batch_seed = ATTACK_SEED + zlib.crc32('|'.join(names).encode('utf-8'))
    torch.manual_seed(batch_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(batch_seed)

    if targets['bboxes'].shape[0] == 0:
        x_adv = x.clone()
    else:
        targets_gpu = {k: v.to(bld.DEVICE) for k, v in targets.items()}
        x_adv = pgd_attack(attack_yolo.model, criterion, x, targets_gpu)

    x_adv_128   = F.interpolate(x_adv, size=bld.IMG_SIZE, mode='bilinear', align_corners=False)
    x_clean_128 = F.interpolate(x,     size=bld.IMG_SIZE, mode='bilinear', align_corners=False)

    with torch.no_grad():
        # A. 干净图直接经过 GCN AE：测量当前重建架构的检测上限。
        z_clean, feats_clean = ae_gcn.encoder(x_clean_128, hard=True)
        x_clean_rec_128 = ae_gcn.decoder(z_clean, feats_clean)

        # B. 攻击图直接经过 GCN AE，不使用 diffusion。
        z_adv, feats_adv = ae_gcn.encoder(x_adv_128, hard=True)
        z_adv_flat = z_adv.view(z_adv.shape[0], -1).long()
        if MODE == "realistic":
            feats_for_decode = feats_adv
        elif MODE == "oracle":
            feats_for_decode = feats_clean
        else:
            raise ValueError(f"未知 MODE: {MODE}")

        x_adv_rec_128 = ae_gcn.decoder(z_adv, feats_for_decode)

        # C. 同一个攻击latent经过 diffusion，再交给同一个 GCN decoder。
        t_vec = torch.full(
            (z_adv_flat.shape[0],), T_STAR - 1,
            device=bld.DEVICE, dtype=torch.long
        )
        z_noised = bld.q_sample(z_adv_flat, t_vec, flip_prob)
        z_rec = bld.reverse(
            diff_model_v3, z_noised, T_STAR, betas, flip_prob
        ).float().view(z_adv.shape)
        x_pur_128 = ae_gcn.decoder(z_rec, feats_for_decode)

    x_clean_rec = F.interpolate(x_clean_rec_128, size=384, mode='bilinear', align_corners=False)
    x_adv_rec = F.interpolate(x_adv_rec_128, size=384, mode='bilinear', align_corners=False)
    x_pur = F.interpolate(x_pur_128, size=384, mode='bilinear', align_corners=False)

    save_as_dfire_split(x,           paths, f'{OUT_ROOT}/original_clean')
    save_as_dfire_split(x_adv,       paths, f'{OUT_ROOT}/adv_original')
    save_as_dfire_split(x_clean_rec, paths, f'{OUT_ROOT}/clean_reconstructed')
    save_as_dfire_split(x_adv_rec,   paths, f'{OUT_ROOT}/adv_reconstructed')
    save_as_dfire_split(x_pur,       paths, f'{OUT_ROOT}/adv_diffusion_purified')

    for n in names:
        progress_f.write(n + '\n')
    progress_f.flush()
    done.update(names)
    processed_batches += 1

progress_f.close()
for handle in gcn_hook_handles:
    handle.remove()

missing_names = eval_names - done
if missing_names:
    raise RuntimeError(f'仍有 {len(missing_names)} 张没有完成重建，停止 mAP')

for split in OUTPUT_SPLITS:
    output_count = len(glob.glob(f'{OUT_ROOT}/{split}/images/*.jpg'))
    print(f'{split:26s} 输出图片数：{output_count}/{EXPECTED_IMAGE_COUNT}')
    if output_count != EXPECTED_IMAGE_COUNT:
        raise RuntimeError(f'{split} 目录图片数量不等于 {EXPECTED_IMAGE_COUNT}，停止 mAP')

print(f"\n全部完成：{len(done)}/{EXPECTED_IMAGE_COUNT} 张均已重建")
print('本次 GCN forward 调用次数:', gcn_hits)
if processed_batches > 0 and not all(v > 0 for v in gcn_hits.values()):
    raise RuntimeError('至少一个 GCN block 没有参与 forward，停止报告 mAP')
if processed_batches == 0:
    print('本次为断点续跑且没有生成新图；mAP 将读取该 checkpoint 标识目录中的已有图片。')

# ==============================================================================
#  3. mAP 评估
# ==============================================================================
print(f"\n=== GCN五路消融 + 真实PGD攻击 [{MODE} mode, T*={T_STAR}] ===")
results = {}
for name in OUTPUT_SPLITS:
    map50, map5095 = eval_map(
        f'{OUT_ROOT}/{name}', img_size=384, detector=detector
    )
    results[name] = (map50, map5095)
    print(f"{name:26s}  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")

recovery50 = (
    results['adv_diffusion_purified'][0] - results['adv_original'][0]
) / max(results['original_clean'][0] - results['adv_original'][0], 1e-8)
recovery5095 = (
    results['adv_diffusion_purified'][1] - results['adv_original'][1]
) / max(results['original_clean'][1] - results['adv_original'][1], 1e-8)
clean_recon_drop = results['original_clean'][0] - results['clean_reconstructed'][0]
direct_recon_gain = results['adv_reconstructed'][0] - results['adv_original'][0]
diffusion_gain = results['adv_diffusion_purified'][0] - results['adv_reconstructed'][0]
print(f"\n净化恢复比例 mAP50 [{MODE}, T*={T_STAR}] = {recovery50:.1%}")
print(f"净化恢复比例 mAP50-95 [{MODE}, T*={T_STAR}] = {recovery5095:.1%}")
print(f"干净重建造成的 mAP50 损失 = {clean_recon_drop:+.4f}")
print(f"攻击图直接重建相对原攻击图的 mAP50 增益 = {direct_recon_gain:+.4f}")
print(f"diffusion 相对直接重建的额外 mAP50 增益 = {diffusion_gain:+.4f}")